In [30]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np


In [41]:
import pandas_market_calendars as mcal
import datetime


nyse = mcal.get_calendar('NYSE')
TOTAL_SECONDS_ONE_YEAR = 365*24*60*60 # total seconds

def get_market_open_close(day_stamp,no_tzinfo=True):
    early = nyse.schedule(start_date=day_stamp, end_date=day_stamp)
    if len(early) == 0:
        raise LookupError("market not open today!")
    market_open = list(early.to_dict()['market_open'].values())[0]
    market_close = list(early.to_dict()['market_close'].values())[0]
    if no_tzinfo:
        return market_open.replace(tzinfo=None),market_close.replace(tzinfo=None)
    else:
        return market_open,market_close

def get_expiry_tstamp(expiry):
    if not isinstance(expiry,str):
        return np.nan
    expiry = datetime.datetime.strptime(expiry,"%Y-%m-%d")
    _,expiry_tstamp = get_market_open_close(expiry)
    return expiry_tstamp.replace(tzinfo=None)

def get_annualized_time_to_expiration(row,expiry_mapper):
    if isinstance(row.expiry,str):
        expiry = row.expiry
    else:
        expiry = row.expiry.strftime("%Y-%m-%d")
    expiry_tstamp = expiry_mapper[expiry]
    sec_to_expiration = (expiry_tstamp-row.tstamp).total_seconds()
    atte = sec_to_expiration/TOTAL_SECONDS_ONE_YEAR
    return atte


In [3]:
cols=['date', 'forward_price', 'tau', 'risk_free_rate', 'is_call', 'strike_price', 'option_price', 'log_moneyness', 'implied_volatility', 'delta', 'time_to_maturity']

def gen_data(dstamp,zero_day_only=True):
    ticker = "SPXW"
    pq_file = f"/mnt/hd1/data/uw-options-cache/SPX/{dstamp}.parquet.gzip"
    df = pd.read_parquet(pq_file)
    df['tstamp_min'] = df.tstamp_sec.apply(lambda x:x.replace(second=0))
    df=df[df.underlying_symbol==ticker]
    print(df.shape)
    if zero_day_only:
        df = df[df.expiry == dstamp]
    print(df.shape)

    expiry_mapper = {x:get_expiry_tstamp(x) for x in df.expiry.unique()}
    df['date']=df.tstamp_min
    df['forward_price']=df.underlying_price
    df['tau']=df.apply(lambda x: get_annualized_time_to_expiration(x,expiry_mapper),axis=1)
    df['risk_free_rate']=1e-7
    df['is_call']=df.option_type.apply(lambda x: 1.0 if x == 'call' else -1.0)
    df['strike_price']=df.strike
    df['option_price']=df.price
    df['log_moneyness']= np.log(df.underlying_price/df.strike)
    # df.implied_volatility
    # df.delta
    df['time_to_maturity']=((df.tau*TOTAL_SECONDS_ONE_YEAR)/(60*60*24)) #???
    #df['time_to_maturity']=((df.tau*TOTAL_SECONDS_ONE_YEAR)/(60*60*24)).astype(int) #???
    
    df = df[(df.tau>0)&(df.log_moneyness.notnull())]
    df = df[cols]
    df['is_ref'] = (np.random.rand(len(df)) > 0.8).astype(int)
    df = df.dropna()
    return df

In [4]:
dstamp = "2025-11-24"
df = gen_data(dstamp)

(1060240, 33)
(875244, 33)


In [5]:
df.head(4)

,date,forward_price,tau,risk_free_rate,is_call,strike_price,option_price,log_moneyness,implied_volatility,delta,time_to_maturity,is_ref
23918,2025-11-24 14:30:00,6636.54,0.000742,1.000000e-07,-1.0,6150.0,0.05,0.076139,0.914931,-0.001079,0.270799,0
23919,2025-11-24 14:30:00,6636.54,0.000742,1.000000e-07,-1.0,6150.0,0.05,0.076139,0.914931,-0.001079,0.270799,0
23930,2025-11-24 14:30:00,6636.54,0.000742,1.000000e-07,-1.0,5600.0,0.05,0.169824,1.921454,-0.000536,0.270799,1
23931,2025-11-24 14:30:00,6636.54,0.000742,1.000000e-07,-1.0,5600.0,0.05,0.169824,1.921454,-0.000536,0.270799,0


In [6]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from data_util import OptionDataset
from hyperiv_util import SetEmbeddingNetwork, HyperNetwork
from trainer_util import trainer
import torch.optim as optim

In [7]:
from inference import load_model,device
model = load_model()

cuda
3 337


In [ ]:
for date in sorted(list(df.date.unique())):
    griddf = df[(df.date==date)&(df.is_ref==0)]
    # ref
    refdf = df[(df.date==date)&(df.is_ref==1)]
    z = refdf[["log_moneyness", "tau", "implied_volatility"]].to_numpy()
    z = z[np.newaxis,:,:]
    # grid
    x = griddf[["log_moneyness", "tau"]].to_numpy()
    x = x[np.newaxis,:,:]

    x = torch.from_numpy(x).to(device).float()
    z = torch.from_numpy(z).to(device).float()
    y_pred = model(z, x).squeeze(-1)
    y_pred = y_pred.cpu().detach().numpy()

    x = x.cpu().detach().numpy()
    z = z.cpu().detach().numpy()

    pdf = pd.DataFrame({
        "log_moneyness":x[:,:,0].squeeze(),
        "tau":x[:,:,1].squeeze(),
        "implied_volatility":y_pred.squeeze()
        })

    plt.subplot(121)
    sns.scatterplot(pdf,x='log_moneyness',y='implied_volatility',hue='tau',s=1,legend=False)
    plt.grid(True)
    plt.subplot(122)
    sns.scatterplot(griddf,x='log_moneyness',y='implied_volatility',hue='tau',s=1,legend=False)
    plt.grid(True)